In [ ]:
import wave
from pathlib import Path
import pandas as pd
import tgt


# helpers

def load_table(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    raise ValueError(f"Unsupported metadata format: {suffix}")


def get_wav_duration(file_path):
    try:
        with wave.open(str(file_path), "rb") as f:
            return f.getnframes() / float(f.getframerate())
    except Exception:
        return 0.0


def get_valid_wavs_with_textgrids(folder):
    return [
        wav_path
        for wav_path in folder.glob("*.wav")
        if wav_path.with_suffix(".TextGrid").exists()
    ]


def count_words_in_textgrid(textgrid_path):
    try:
        tg = tgt.read_textgrid(str(textgrid_path))
        tier = tg.get_tier_by_name("words")
    except Exception:
        return 0

    return sum(
        1
        for interval in tier.intervals
        if str(interval.text).strip().lower()
        not in {"", "sil", "sp", "spn", "<sil>", "silence"}
    )


def get_folder_stats(folder, max_context_words=10):
    valid_wavs = get_valid_wavs_with_textgrids(folder)

    total_duration = 0.0
    total_words = 0
    total_utterances = 0

    for wav_path in valid_wavs:
        tg_path = wav_path.with_suffix(".TextGrid")

        dur = get_wav_duration(wav_path)
        n_words = count_words_in_textgrid(tg_path)

        if dur > 0 and n_words > 0:
            total_utterances += 1
            total_duration += dur
            total_words += n_words


    return total_duration, total_words, total_utterances


# labels

def normalize_gender(value):
    if pd.isna(value):
        return pd.NA
    return str(value).strip().lower()


def score_to_label(score):
    if pd.isna(score):
        return None
    return "D" if float(score) >= 10 else "H"


def score_to_depression_severity(score):
    if pd.isna(score):
        return pd.NA
    score = float(score)
    if score < 10:
        return 0
    elif score <= 19:
        return 1
    else:
        return 2


def assign_status(meta, participant_col):
    meta = meta.copy()

    meta["Depression_severity"] = pd.to_numeric(
        meta["Depression_severity"], errors="coerce"
    )

    meta["label_depression"] = meta["Depression_severity"].apply(score_to_label)
    meta["depression_severity"] = meta["Depression_severity"].apply(
        score_to_depression_severity
    )
    meta["gender"] = meta["gender"].apply(normalize_gender)

    return dict(
        zip(
            meta[participant_col].astype(str),
            zip(
                meta["label_depression"],
                meta["depression_severity"],
                meta["gender"],
            ),
        )
    )


# splitting logic

def assign_balanced_splits(
    df,
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    utterance_weight=50.0,
    participant_weight=25.0,
    depression_weight=50.0,
    gender_weight=10.0,
    word_weight=3.0,
    duration_weight=1.0,
    soft_cap_factor=1.05,
):
    splits = ["train", "val", "test"]
    ratios = {"train": train_ratio, "val": val_ratio, "test": test_ratio}

    gender_labels = sorted(df["gender"].dropna().unique())
    dep_labels = sorted(df["label_depression"].dropna().unique())

    speakers = []
    for pid, sub in df.groupby("participant_id"):
        speakers.append({
            "pid": pid,
            "utterances": float(sub["num_utterances"].sum()),
            "words": float(sub["num_words"].sum()),
            "duration": float(sub["duration"].sum()),
            "gender": sub["gender"].iloc[0],
            "dep": sub["label_depression"].iloc[0],
        })

    speakers.sort(key=lambda x: x["utterances"], reverse=True)

    totals = {
        "utterances": float(df["num_utterances"].sum()),
        "participants": float(df["participant_id"].nunique()),
        "words": float(df["num_words"].sum()),
        "duration": float(df["duration"].sum()),
    }

    total_gender = {g: sum(sp["utterances"] for sp in speakers if sp["gender"] == g) for g in gender_labels}
    total_dep = {d: sum(sp["utterances"] for sp in speakers if sp["dep"] == d) for d in dep_labels}

    targets = {s: {k: v * ratios[s] for k, v in totals.items()} for s in splits}
    target_gender = {s: {g: total_gender[g] * ratios[s] for g in gender_labels} for s in splits}
    target_dep = {s: {d: total_dep[d] * ratios[s] for d in dep_labels} for s in splits}

    current = {s: {k: 0.0 for k in totals} for s in splits}
    current_gender = {s: {g: 0.0 for g in gender_labels} for s in splits}
    current_dep = {s: {d: 0.0 for d in dep_labels} for s in splits}

    assignment = {}

    def add(split, sp):
        assignment[sp["pid"]] = split
        current[split]["utterances"] += sp["utterances"]
        current[split]["participants"] += 1
        current[split]["words"] += sp["words"]
        current[split]["duration"] += sp["duration"]
        current_gender[split][sp["gender"]] += sp["utterances"]
        current_dep[split][sp["dep"]] += sp["utterances"]

    def cost(split, sp):
        eps = 1e-8

        after_utt = current[split]["utterances"] + sp["utterances"]
        target_utt = targets[split]["utterances"] + eps

        # Hard-ish cap for val/test so one large speaker cannot dominate.
        if split in {"val", "test"} and after_utt > soft_cap_factor * target_utt:
            return 1e12 + after_utt

        c = 0.0

        c += utterance_weight * ((after_utt - target_utt) / target_utt) ** 2

        after_part = current[split]["participants"] + 1
        target_part = targets[split]["participants"] + eps
        c += participant_weight * ((after_part - target_part) / target_part) ** 2

        for key, weight in [
            ("words", word_weight),
            ("duration", duration_weight),
        ]:
            after = current[split][key] + sp[key]
            target = targets[split][key] + eps
            c += weight * ((after - target) / target) ** 2

        g = sp["gender"]
        if g in gender_labels:
            after = current_gender[split][g] + sp["utterances"]
            target = target_gender[split][g] + eps
            c += gender_weight * ((after - target) / target) ** 2

        d = sp["dep"]
        if d in dep_labels:
            after = current_dep[split][d] + sp["utterances"]
            target = target_dep[split][d] + eps

            # Prevent val/test from becoming dominated by one label
            if split in {"val", "test"} and after > 1.15 * target:
                return 1e12 + after

            c += 100.0 * ((after - target) / target) ** 2

        return c

    # Fill val/test first to hit utterance targets.
    remaining = []
    for sp in speakers:
        candidate_splits = ["val", "test", "train"]
        best = min(candidate_splits, key=lambda s: cost(s, sp))
        add(best, sp)

    out = df.copy()
    out["split"] = out["participant_id"].map(assignment)
    return out



def create_balanced_splits(
    mfa_root,
    metadata_path,
    output_csv,
    participant_col,
):
    root = Path(mfa_root)
    meta = load_table(metadata_path)

    status_map = assign_status(meta, participant_col)

    data = []

    for folder in root.iterdir():
        if not folder.is_dir():
            continue

        pid = folder.name
        if pid not in status_map:
            continue

        label_dep, dep_sev, gender = status_map[pid]

        if label_dep is None or pd.isna(gender):
            continue

        dur, words, utts = get_folder_stats(folder)

        if utts == 0:
            continue

        data.append(
            {
                "participant_id": pid,
                "gender": gender,
                "label_depression": label_dep,
                "depression_severity": dep_sev,
                "duration": dur,
                "num_words": words,
                "num_utterances": utts,
            }
        )

    df = pd.DataFrame(data)

    df = assign_balanced_splits(df)

    df.to_csv(output_csv, index=False)

    print("\nUtterances per split:")
    print(df.groupby("split")["num_utterances"].sum())

    print("\nParticipants per split:")
    print(df.groupby("split")["participant_id"].nunique())

    print("\nDepression balance:")
    print(df.groupby(["split", "label_depression"]).size())

    print("\nGender counts by split (participants):")
    gender_participants = (
        df.groupby(["split", "participant_id"])["gender"]
        .first()
        .reset_index()
        .groupby(["split", "gender"])["participant_id"]
        .nunique()
    )
    print(gender_participants)

    print("\nGender counts by split (utterances):")
    gender_utterances = df.groupby(["split", "gender"])["num_utterances"].sum()
    print(gender_utterances)

    print("\nWord counts by split:")
    gender_utterances = df.groupby(["split"])["num_words"].sum()
    print(gender_utterances)


    print("\nGender proportions by split (utterances):")
    gender_props = (
        df.groupby(["split", "gender"])["num_utterances"].sum()
        / df.groupby("split")["num_utterances"].sum()
    )
    print(gender_props.round(3))

In [5]:
create_balanced_splits(
    mfa_root="/work/DISCOURSE/Data/DAIC-WOZ/Aligned_Data",
    metadata_path="/work/DISCOURSE/Data/DAIC-WOZ/Metadata/DAIC_participant_data.csv",
    output_csv="DAIC_splits.csv",
    participant_col="Participant",
)


Utterances per split:
split
test      714
train    5376
val       711
Name: num_utterances, dtype: int64

Participants per split:
split
test      18
train    225
val       15
Name: participant_id, dtype: int64

Depression balance:
split  label_depression
test   D                     5
       H                    13
train  D                    73
       H                   152
val    D                     5
       H                    10
dtype: int64

Gender counts by split (participants):
split  gender 
test   female       8
       male        10
train  female      86
       male       138
       unknown      1
val    female       4
       male        11
Name: participant_id, dtype: int64

Gender counts by split (utterances):
split  gender 
test   female      313
       male        401
train  female     2220
       male       3140
       unknown      16
val    female      206
       male        505
Name: num_utterances, dtype: int64

Word counts by split:
split
test      30900
train  